# 💳 Parcelado sem juros é de graça?
## O custo que o lojista embutiu no preço — e você não viu
### Edição 2025 — dados reais IBGE · BACEN · adquirentes

---

> **Spoiler:** Não é de graça. O custo existe — ele só foi transferido para o preço.

Este notebook mostra:
1. O custo real do parcelamento via MDR / taxa de antecipação
2. A corrosão pelo IPCA no período das parcelas
3. O que o lojista sabe (e precifica) que o consumidor ignora
4. **Dois modelos economicamente corretos** para responder: *vale parcelar e investir ou pagar à vista?*

**Dados reais 2025:**
- IPCA mensal jan–dez 2025 (IBGE)
- Selic Over acumulada 2025: **14,32% a.a.** (BACEN)
- MDR crédito parcelado: **1,3% a 5,9%** do valor da venda (BACEN / adquirentes)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f8f6',
    'axes.grid': True, 'grid.color': '#e0e0d8', 'grid.linewidth': 0.7,
    'font.family': 'DejaVu Sans', 'axes.spines.top': False,
    'axes.spines.right': False, 'axes.titlesize': 13, 'axes.labelsize': 11,
})

# ── Parâmetros globais ──────────────────────────────────────────────────
PRECO_PRODUTO   = 3_000.0          # custo real que o lojista quer receber
SELIC_AA_2025   = 0.1432           # Selic Over acumulada 2025 (BACEN)
CDI_MENSAL      = (1 + SELIC_AA_2025)**(1/12) - 1

MDR = {2:  {'min':3.5,  'med':4.5,  'max':6.0},
       3:  {'min':4.0,  'med':5.5,  'max':7.5},
       4:  {'min':4.8,  'med':6.5,  'max':8.5},
       6:  {'min':6.5,  'med':8.5,  'max':11.0},
       8:  {'min':8.0,  'med':11.0, 'max':14.0},
       10: {'min':9.5,  'med':13.0, 'max':17.0},
       12: {'min':11.0, 'med':15.0, 'max':20.0}}

IPCA_2025 = {'Jan':0.42,'Fev':1.31,'Mar':0.56,'Abr':0.43,
             'Mai':0.26,'Jun':0.24,'Jul':0.26,'Ago':-0.11,
             'Set':0.48,'Out':0.09,'Nov':-0.09,'Dez':0.33}

print(f'CDI mensal 2025: {CDI_MENSAL*100:.4f}%  |  CDI anual: {SELIC_AA_2025*100:.2f}%')
print('✅ Parâmetros carregados.')

---
## 📌 Parte 1 — Custo MDR: o que o lojista paga para oferecer parcelamento

A adquirente antecipa o valor da venda parcelada ao lojista — mas desconta o **MDR** (*Merchant Discount Rate*). Com a Selic a 15% em 2025, o custo de antecipação de recebíveis ficou ainda mais alto.

In [ ]:
parcelas_lista = list(MDR.keys())
rows = []
for n, t in MDR.items():
    rows.append({'Parcelas': f'{n}x',
                 'Custo mín (R$)': f"R$ {PRECO_PRODUTO*t['min']/100:,.2f}",
                 'Custo méd (R$)': f"R$ {PRECO_PRODUTO*t['med']/100:,.2f}",
                 'Custo máx (R$)': f"R$ {PRECO_PRODUTO*t['max']/100:,.2f}",
                 '% méd do preço': f"{t['med']:.1f}%"})
print(f'Produto: R$ {PRECO_PRODUTO:,.0f} — MDR total por número de parcelas (2025)\n')
print(pd.DataFrame(rows).to_string(index=False))
print('\n⚠️  Este custo é pago pelo lojista — e embutido no preço de todos os produtos.')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Custo real do parcelamento para o lojista — MDR 2025', fontsize=14, fontweight='bold')

pcts_min = [MDR[n]['min'] for n in parcelas_lista]
pcts_med = [MDR[n]['med'] for n in parcelas_lista]
pcts_max = [MDR[n]['max'] for n in parcelas_lista]
custos_med = [PRECO_PRODUTO*p/100 for p in pcts_med]
custos_min = [PRECO_PRODUTO*p/100 for p in pcts_min]
custos_max = [PRECO_PRODUTO*p/100 for p in pcts_max]
x = np.arange(len(parcelas_lista))
labels = [f'{n}x' for n in parcelas_lista]

bars = ax1.bar(x, custos_med, color='#2a78d6', alpha=0.85, width=0.5, label='MDR médio', zorder=3)
ax1.vlines(x, custos_min, custos_max, color='#444', linewidth=2.5, zorder=4)
ax1.scatter(x, custos_min, color='#1baf7a', zorder=5, s=55)
ax1.scatter(x, custos_max, color='#e34948', zorder=5, s=55)
ax1.set_xticks(x); ax1.set_xticklabels(labels)
ax1.set_ylabel('Custo (R$)'); ax1.set_xlabel('Parcelas')
ax1.set_title('Custo absoluto (produto R$ 3.000)')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'R$ {v:,.0f}'))
for bar, val in zip(bars, custos_med):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
             f'R${val:.0f}', ha='center', va='bottom', fontsize=8.5, color='#1a5ba8')

ax2.fill_between(parcelas_lista, pcts_min, pcts_max, alpha=0.18, color='#2a78d6')
ax2.plot(parcelas_lista, pcts_med, 'o-', color='#2a78d6', linewidth=2.5, markersize=7, label='Méd')
ax2.plot(parcelas_lista, pcts_min, '--', color='#1baf7a', linewidth=1.5, alpha=0.85, label='Mín')
ax2.plot(parcelas_lista, pcts_max, '--', color='#e34948', linewidth=1.5, alpha=0.85, label='Máx')
ax2.set_xlabel('Parcelas'); ax2.set_ylabel('% do preço')
ax2.set_title('Custo como % do preço')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:.1f}%'))
for n, p in zip(parcelas_lista, pcts_med):
    ax2.text(n, p+0.4, f'{p:.1f}%', ha='center', fontsize=8.5, color='#1a5ba8')
ax2.legend(fontsize=9)

plt.tight_layout(); plt.savefig('g1_mdr_2025.png', dpi=150, bbox_inches='tight'); plt.show()

---
## 📌 Parte 2 — IPCA 2025: dois meses de deflação

2025 teve deflação em **agosto (−0,11%)** e **novembro (−0,09%)** — incomum no Brasil. O IPCA fechou o ano em **4,26%**, abaixo do teto da meta (4,5%) e o menor desde 2018 (IBGE).

In [ ]:
meses     = list(IPCA_2025.keys())
ipca_vals = list(IPCA_2025.values())
ipca_acum = np.cumprod([1 + v/100 for v in ipca_vals]) - 1
cdi_acum  = np.cumprod([1 + CDI_MENSAL]*12) - 1

df_inf = pd.DataFrame({'Mês': meses, 'IPCA (%)': ipca_vals,
                        'CDI mensal (%)': [f'{CDI_MENSAL*100:.3f}']*12,
                        'IPCA acum. (%)': [f'{v*100:.2f}' for v in ipca_acum],
                        'CDI acum. (%)':  [f'{v*100:.2f}' for v in cdi_acum]})
print(df_inf.to_string(index=False))
print(f'\nIPCA acumulado 2025: {ipca_acum[-1]*100:.2f}%  |  CDI acumulado: {cdi_acum[-1]*100:.2f}%')
print(f'Rentabilidade REAL do CDI em 2025: {((1+cdi_acum[-1])/(1+ipca_acum[-1])-1)*100:.2f}%')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('IPCA 2025 (IBGE) — inflação mensal e erosão do valor real das parcelas', fontsize=14, fontweight='bold')

cores_b = ['#e34948' if v >= 0 else '#1baf7a' for v in ipca_vals]
ax1.bar(range(12), ipca_vals, color=cores_b, alpha=0.88, zorder=3)
ax1.axhline(0, color='#555', linewidth=0.9)
ax1.set_xticks(range(12)); ax1.set_xticklabels(meses, fontsize=9)
ax1.set_ylabel('IPCA mensal (%)'); ax1.set_title('IPCA mensal 2025 — ago e nov com deflação')
for i, v in enumerate(ipca_vals):
    ax1.text(i, v + (0.04 if v >= 0 else -0.12), f'{v:+.2f}%',
             ha='center', fontsize=7.5, color='#a32d2d' if v >= 0 else '#0d7a52')
ax1.legend(handles=[mpatches.Patch(color='#e34948', label='Inflação'),
                    mpatches.Patch(color='#1baf7a', label='Deflação')], fontsize=9)

parcela_nominal = PRECO_PRODUTO / 12
vals_reais = [parcela_nominal / (1 + v) for v in ipca_acum]
vals_oport = [parcela_nominal / (1 + v) for v in cdi_acum]
ax2.axhline(parcela_nominal, color='#888', linestyle='--', linewidth=1.2,
            label=f'Nominal: R$ {parcela_nominal:.2f}', alpha=0.75)
ax2.plot(meses, vals_reais, 'o-', color='#e34948', linewidth=2.2, markersize=6, label='Descontado IPCA')
ax2.plot(meses, vals_oport, 's--', color='#2a78d6', linewidth=1.8, markersize=5, label='Descontado CDI')
ax2.fill_between(range(12), vals_reais, parcela_nominal, alpha=0.1, color='#e34948')
ax2.set_title(f'Valor real de cada parcela R$ {parcela_nominal:.0f} (12x, 2025)')
ax2.set_ylabel('R$'); ax2.set_xticks(range(12)); ax2.set_xticklabels(meses, fontsize=9)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'R$ {v:.2f}'))
ax2.legend(fontsize=9)

plt.tight_layout(); plt.savefig('g2_ipca_2025.png', dpi=150, bbox_inches='tight'); plt.show()

---
## 📌 Parte 3 — O sobrepreço embutido pelo lojista

In [ ]:
precos = [PRECO_PRODUTO / (1 - MDR[n]['med']/100) for n in parcelas_lista]
sobres = [p - PRECO_PRODUTO for p in precos]

rows_p = [{'Parcelas': f'{n}x',
           'Preço anunciado': f'R$ {p:,.2f}',
           'Sobrepreço': f'R$ {s:,.2f}',
           '% acima do custo': f'{s/PRECO_PRODUTO*100:.1f}%'}
          for n, p, s in zip(parcelas_lista, precos, sobres)]
print('Para receber R$ 3.000 líquidos, o lojista precisa cobrar:\n')
print(pd.DataFrame(rows_p).to_string(index=False))
print('\n🔍 Quem paga à vista sem pedir desconto paga o MDR dos parceladores.')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Sobrepreço embutido para bancar o "sem juros" — 2025', fontsize=14, fontweight='bold')

x = np.arange(len(parcelas_lista))
ax1.bar(x, [PRECO_PRODUTO]*len(x), color='#2a78d6', label='Custo real', width=0.5)
ax1.bar(x, sobres, bottom=[PRECO_PRODUTO]*len(x), color='#e34948', label='Sobrepreço (MDR)', width=0.5, alpha=0.9)
ax1.set_xticks(x); ax1.set_xticklabels([f'{n}x' for n in parcelas_lista])
ax1.set_ylabel('R$'); ax1.set_title('Decomposição do preço anunciado')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'R$ {v:,.0f}'))
ax1.legend(fontsize=9)
for i, s in enumerate(sobres):
    ax1.text(i, PRECO_PRODUTO+s+8, f'+R${s:.0f}', ha='center', fontsize=8, color='#a32d2d', fontweight='bold')

preco_12 = precos[-1]  # 12x
sobre_12 = sobres[-1]
perfis = ['À vista\n(sem pedir desconto)', 'À vista\n(8% desconto)', 'Parcela 12x\n"sem juros"']
vals_p = [preco_12, preco_12*0.92, preco_12]
cores_p = ['#e34948', '#1baf7a', '#2a78d6']
bars3 = ax2.barh(perfis, vals_p, color=cores_p, alpha=0.85, height=0.45)
ax2.axvline(PRECO_PRODUTO, color='#555', linestyle='--', linewidth=1.5,
            label=f'Custo real: R$ {PRECO_PRODUTO:,.0f}')
ax2.set_xlabel('R$ pago')
ax2.set_title(f'Cenário 12x — quem paga mais?')
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'R$ {v:,.0f}'))
for bar, val in zip(bars3, vals_p):
    ax2.text(val+3, bar.get_y()+bar.get_height()/2, f'R$ {val:,.0f}', va='center', fontsize=9)
ax2.legend(fontsize=9)

plt.tight_layout(); plt.savefig('g3_sobrepreco_2025.png', dpi=150, bbox_inches='tight'); plt.show()

---
## 📌 Parte 4 — Vale parcelar e investir ou pagar à vista?

Esta é a pergunta clássica de finanças pessoais — e a resposta depende do modelo de fluxo de caixa usado.

---

### Modelo A — Mesmo capital inicial para os dois cenários

Ambos começam com **R$ 3.529,41** (o preço cheio em 12x).

| Quem | O que faz |
|---|---|
| **Parcelado** | Investe R$ 3.529,41. A cada mês, o saldo rende CDI e saca R$ 294,12 para pagar a parcela. |
| **À vista** | Paga R$ 3.247,06 hoje (8% desconto). Investe apenas o troco: R$ 282,35. |

### Modelo B — Clássico finanças pessoais

A pergunta é: **os juros obtidos enquanto o dinheiro fica investido compensam o desconto à vista?**

- Parcelado: mantém R$ 3.529,41 investido, saca R$ 294,12/mês → **juros obtidos = saldo final**
- À vista: recebe o **desconto** imediatamente
- O break-even é o desconto exato que iguala os juros obtidos


In [ ]:
preco_12     = precos[-1]          # R$ 3.529,41
parcela_12x  = preco_12 / 12      # R$ 294,12
desc_pct     = 0.08
preco_vista  = preco_12 * (1 - desc_pct)
troco        = preco_12 - preco_vista  # R$ 282,35

# ── Modelo A ─────────────────────────────────────────────────────────
saldo_parc = preco_12
saldo_parc_mes = []
for i in range(12):
    saldo_parc = saldo_parc * (1 + CDI_MENSAL) - parcela_12x
    saldo_parc_mes.append(saldo_parc)

saldo_vista_mes = [troco * (1 + CDI_MENSAL)**(i+1) for i in range(12)]
saldo_vista_final = saldo_vista_mes[-1]
vantagem_A = saldo_parc_mes[-1] - saldo_vista_final

# ── Modelo B ─────────────────────────────────────────────────────────
# Juros obtidos parcelando = saldo final após todos os saques
juros_parcelado = saldo_parc_mes[-1]
desconto_obtido = troco
be_pct = juros_parcelado / preco_12 * 100
vantagem_B = desconto_obtido - juros_parcelado

print('='*58)
print('MODELO A — Mesmo capital inicial (R$ 3.529,41)')
print('='*58)
print(f'  Parcelado: investe R$ {preco_12:,.2f}, saca R$ {parcela_12x:.2f}/mês')
print(f'  Saldo final (parcelado):  R$ {saldo_parc_mes[-1]:,.2f}')
print(f'  À vista: paga R$ {preco_vista:,.2f}, investe troco R$ {troco:.2f}')
print(f'  Saldo final (à vista):    R$ {saldo_vista_final:,.2f}')
print(f'  Vantagem: {"PARCELADO" if vantagem_A > 0 else "À VISTA"} por R$ {abs(vantagem_A):.2f}')

print()
print('='*58)
print('MODELO B — Clássico finanças pessoais')
print('='*58)
print(f'  Juros obtidos mantendo o capital investido: R$ {juros_parcelado:.2f}')
print(f'  ({juros_parcelado/preco_12*100:.2f}% do preço cheio)')
print(f'  Desconto obtido pagando à vista (8%):      R$ {desconto_obtido:.2f}')
print(f'\n  Break-even: desconto de {be_pct:.2f}%')
print(f'  → Abaixo de {be_pct:.2f}%: PARCELAR e investir é melhor')
print(f'  → Acima de {be_pct:.2f}%:  PAGAR À VISTA é melhor')
print(f'\n  Com 8% de desconto → PAGAR À VISTA vence por R$ {vantagem_B:.2f}')

print()
print('Evolução mês a mês (Modelo B — saldo enquanto parcela):')
print(f'{"Mês":>4} | {"Saldo início":>14} | {"Juros CDI":>10} | {"Saque parcela":>14} | {"Saldo fim":>12}')
saldo = preco_12
for i in range(12):
    j = saldo * CDI_MENSAL
    saldo_novo = saldo + j - parcela_12x
    print(f'{i+1:>4} | R$ {saldo:>11,.2f} | R$ {j:>7,.2f} | R$ {parcela_12x:>11,.2f} | R$ {saldo_novo:>9,.2f}')
    saldo = saldo_novo

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Parcelar e investir vs. pagar à vista — 2025 (CDI 14,32% a.a.)', fontsize=14, fontweight='bold')

meses_num = list(range(1, 13))

# Gráfico A — Saldo mês a mês (Modelo A)
ax = axes[0]
ax.plot(meses_num, saldo_parc_mes, 'o-', color='#2a78d6', linewidth=2.3, markersize=6,
        label=f'Parcelado (saldo após saque)')
ax.plot(meses_num, saldo_vista_mes, 's--', color='#1baf7a', linewidth=2, markersize=6,
        label=f'À vista (troco R${troco:.0f} investido)')
ax.axhline(0, color='#aaa', linewidth=0.8)
ax.set_title('Modelo A — Saldo final\n(mesmo capital inicial)', fontsize=11)
ax.set_xlabel('Mês'); ax.set_ylabel('R$')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'R$ {v:,.0f}'))
ax.legend(fontsize=8.5); ax.set_xticks(meses_num)
ax.text(12, saldo_parc_mes[-1]+10, f'R$ {saldo_parc_mes[-1]:.0f}', ha='right', fontsize=8.5, color='#2a78d6')
ax.text(12, saldo_vista_mes[-1]+10, f'R$ {saldo_vista_mes[-1]:.0f}', ha='right', fontsize=8.5, color='#1baf7a')

# Gráfico B — Saldo decrescente (Modelo B)
ax2 = axes[1]
ax2.plot(meses_num, saldo_parc_mes, 'o-', color='#2a78d6', linewidth=2.3, markersize=6,
         label='Capital investido (após saque)')
ax2.fill_between(meses_num, 0, saldo_parc_mes, alpha=0.12, color='#2a78d6')
# Área de juros acumulados
saldo_sem_juros = [preco_12 - parcela_12x*(i+1) for i in range(12)]
ax2.plot(meses_num, saldo_sem_juros, '--', color='#888', linewidth=1.5, label='Sem juros (referência)')
ax2.fill_between(meses_num, saldo_sem_juros, saldo_parc_mes, alpha=0.2, color='#f5a623',
                 label=f'Juros CDI ganhos')
ax2.set_title('Modelo B — Saldo decrescente\n(juros ganhos ao longo do ano)', fontsize=11)
ax2.set_xlabel('Mês'); ax2.set_ylabel('R$')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'R$ {v:,.0f}'))
ax2.legend(fontsize=8.5); ax2.set_xticks(meses_num)
ax2.text(6, 1700, f'Juros totais\nR$ {juros_parcelado:.0f}', ha='center', fontsize=9,
         color='#b8750a', fontweight='bold')

# Gráfico C — Break-even: desconto mínimo para valer à vista
ax3 = axes[2]
descontos_test = np.linspace(0, 20, 200)
# Para cada desconto: ganho à vista = preco_12 * d%
# Ganho parcelado = juros_parcelado (fixo)
vantagens = [preco_12*d/100 - juros_parcelado for d in descontos_test]
ax3.plot(descontos_test, vantagens, color='#2a78d6', linewidth=2.5)
ax3.axhline(0, color='#555', linewidth=1)
ax3.axvline(be_pct, color='#e34948', linewidth=2, linestyle='--', label=f'Break-even: {be_pct:.2f}%')
ax3.axvline(8, color='#1baf7a', linewidth=2, linestyle=':', label='Desconto obtido: 8%')
ax3.fill_between(descontos_test, vantagens, 0,
                 where=[v > 0 for v in vantagens], alpha=0.15, color='#1baf7a',
                 label='À vista é melhor')
ax3.fill_between(descontos_test, vantagens, 0,
                 where=[v < 0 for v in vantagens], alpha=0.15, color='#e34948',
                 label='Parcelar é melhor')
ax3.set_xlabel('Desconto à vista (%)'); ax3.set_ylabel('Vantagem à vista (R$)')
ax3.set_title(f'Break-even do desconto\n(CDI 14,32% a.a. — 12x)', fontsize=11)
ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'R$ {v:,.0f}'))
ax3.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:.0f}%'))
ax3.legend(fontsize=8.5)
ax3.text(be_pct+0.3, -80, f'{be_pct:.2f}%', color='#e34948', fontsize=9, fontweight='bold')

plt.tight_layout(); plt.savefig('g4_modelos_2025.png', dpi=150, bbox_inches='tight'); plt.show()

print(f'\n✅ Break-even confirmado: {be_pct:.2f}%')
print(f'   Com 8% de desconto → à vista vence por R$ {vantagem_B:.2f}')
print(f'   Com menos de {be_pct:.2f}% → parcelar e investir é melhor')

---
## 📌 Parte 5 — Painel consolidado

In [ ]:
fig = plt.figure(figsize=(14, 10))
fig.suptitle('Parcelado sem juros — todos os custos ocultos · 2025', fontsize=15, fontweight='bold', y=1.01)
gs = GridSpec(2, 3, figure=fig, hspace=0.48, wspace=0.38)

# ① MDR por parcelas
a1 = fig.add_subplot(gs[0, 0])
a1.plot(parcelas_lista, pcts_med, 'o-', color='#2a78d6', linewidth=2.5, markersize=8)
a1.fill_between(parcelas_lista, pcts_med, alpha=0.12, color='#2a78d6')
a1.set_title('① Custo MDR por parcelas', fontsize=11)
a1.set_xlabel('Parcelas'); a1.set_ylabel('% do preço')
a1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:.1f}%'))
for n, p in zip(parcelas_lista, pcts_med): a1.text(n, p+0.3, f'{p:.1f}%', ha='center', fontsize=8)

# ② IPCA 2025
a2 = fig.add_subplot(gs[0, 1])
a2.bar(range(12), ipca_vals, color=['#e34948' if v>=0 else '#1baf7a' for v in ipca_vals], alpha=0.85, zorder=3)
a2.axhline(0, color='#555', linewidth=0.8)
a2.set_xticks(range(12)); a2.set_xticklabels(meses, fontsize=7.5)
a2.set_title('② IPCA mensal 2025\n(ago e nov = deflação)', fontsize=11); a2.set_ylabel('%')

# ③ Sobrepreço
a3 = fig.add_subplot(gs[0, 2])
a3.bar(range(len(parcelas_lista)), [PRECO_PRODUTO]*len(parcelas_lista), color='#2a78d6', label='Custo real', width=0.5)
a3.bar(range(len(parcelas_lista)), sobres, bottom=[PRECO_PRODUTO]*len(parcelas_lista),
       color='#e34948', label='Sobrepreço', width=0.5, alpha=0.9)
a3.set_xticks(range(len(parcelas_lista))); a3.set_xticklabels([f'{n}x' for n in parcelas_lista], fontsize=8)
a3.set_title('③ Sobrepreço embutido', fontsize=11); a3.set_ylabel('R$'); a3.legend(fontsize=8)
a3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'R${v:.0f}'))

# ④ Break-even
a4 = fig.add_subplot(gs[1, 0])
a4.plot(descontos_test, vantagens, color='#2a78d6', linewidth=2.5)
a4.axhline(0, color='#555', linewidth=1)
a4.axvline(be_pct, color='#e34948', linewidth=2, linestyle='--', label=f'Break-even {be_pct:.2f}%')
a4.axvline(8, color='#1baf7a', linewidth=2, linestyle=':', label='Desconto obtido 8%')
a4.fill_between(descontos_test, vantagens, 0, where=[v>0 for v in vantagens], alpha=0.15, color='#1baf7a')
a4.fill_between(descontos_test, vantagens, 0, where=[v<0 for v in vantagens], alpha=0.15, color='#e34948')
a4.set_title('④ Break-even do desconto\n(12x, CDI 14,32%)', fontsize=11)
a4.set_xlabel('Desconto (%)'); a4.set_ylabel('Vantagem à vista (R$)')
a4.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'R${v:,.0f}'))
a4.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:.0f}%'))
a4.legend(fontsize=8)

# ⑤ Custo efetivo por estratégia (Modelo A)
a5 = fig.add_subplot(gs[1, 1:])
cenarios = ['Parcela 12x\n(sem investir)', 'À vista 8% desc.\n(investe R$282)', 'Parcela 12x\n(investe e saca CDI)']
# Cenário 1: paga 12x sem investir → gasta preco_12, saldo 0
# Cenário 2: paga à vista com desc., investe troco
# Cenário 3: investe tudo, saca para pagar → saldo_parc_mes[-1]
vals_cen = [0, saldo_vista_final, saldo_parc_mes[-1]]
custo_cen = [preco_12, preco_vista, preco_12]
cores_cen = ['#e34948', '#1baf7a', '#2a78d6']
x5 = np.arange(3)
bars_custo = a5.bar(x5-0.2, custo_cen, width=0.35, color=cores_cen, alpha=0.5, label='Desembolso')
bars_saldo = a5.bar(x5+0.2, vals_cen, width=0.35, color=cores_cen, alpha=0.95, label='Saldo final investido')
a5.axhline(PRECO_PRODUTO, linestyle='--', color='#555', linewidth=1.3, label=f'Custo real: R$ {PRECO_PRODUTO:,.0f}')
a5.set_xticks(x5); a5.set_xticklabels(cenarios, fontsize=9)
a5.set_ylabel('R$'); a5.set_title('⑤ Desembolso vs saldo final por estratégia (Modelo A)', fontsize=11)
a5.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'R$ {v:,.0f}'))
a5.legend(fontsize=9)
for bar, val in zip(bars_saldo, vals_cen):
    if val > 0:
        a5.text(bar.get_x()+bar.get_width()/2, val+15, f'R$ {val:.0f}',
                ha='center', fontsize=9, fontweight='bold')

plt.savefig('g5_painel_2025.png', dpi=150, bbox_inches='tight'); plt.show()
print('✅ Painel final salvo.')

---
## 📋 Resumo Executivo — 2025

| Pergunta | Resposta |
|---|---|
| **O MDR custa quanto ao lojista em 12x?** | ~15% do valor da venda (médio 2025). |
| **Quem paga esse custo?** | Todos, via sobrepreço embutido. À vista sem pedir desconto = subsidia os parceladores. |
| **Break-even: qual desconto torna à vista melhor?** | **7,92%** do preço cheio (com CDI a 14,32% e 12x). |
| **Com 8% de desconto, o que é melhor?** | Pagar à vista vence por **R$ 2,99** — margem mínima. |
| **Se não conseguir desconto > 7,92%?** | Parcelar e manter o capital investido no CDI é matematicamente melhor. |

> **Insight central:** em 2025, com CDI a 14,32%, o break-even caiu para ~7,92%. Isso significa que o parcelamento sem juros ficou **mais atraente** do que em anos de Selic baixa — porque o dinheiro rende mais enquanto está investido. A "gratuidade" do parcelamento é real só se você realmente investir o capital e tiver disciplina para sacar apenas o valor das parcelas.

---

### Fontes
- **IBGE / Agência de Notícias** — IPCA mensal jan–dez 2025
- **Banco Central do Brasil (BACEN)** — Selic Over acumulada 2025 (14,32% a.a.), MDR e intercâmbio
- **Copom / BACEN** — Selic meta: chegou a 15% a.a. em jun/2025
- **Adquirentes (Cielo, Stone, PagSeguro, GetNet)** — tabelas públicas MDR 2025
- **ABECS** — Associação Brasileira das Empresas de Cartões de Crédito e Serviços